# Evaluación de generación Cisco — 6 modelos base (float16)
**Modelos:** Llama-3.1-8B, Zephyr-7B, Qwen2.5-7B, Gemma-2-9B (causal) + FLAN-T5-large, FLAN-T5-base (seq2seq)
**Carga:** `torch_dtype=torch.float16`, sin cuantización (un modelo a la vez, liberando VRAM entre cada uno).
**Dataset:** `eval_dataset_150_cli_corrected.csv` (columnas `requirement`, `configuration`).
**Flujo:** generación directa o con plan (`USE_PLANNING`). Prompts y métricas propias del notebook.

**Métricas:** ROUGE-1/2/L normalizado (sin prompts CLI) + BERTScore | **GPU:** A100 recomendada

---
### Checklist antes de ejecutar
1. Menú → **Entorno de ejecución → Cambiar tipo de entorno** → GPU
2. Sube `eval_dataset_150_cli_corrected.csv` a `/content/drive/MyDrive/eval modelos base`
3. Agrega tu `HF_TOKEN` en Secrets (ícono 🔑) — con acceso aceptado a Llama y Gemma
4. Ajusta flags en **Celda 2 — Configuración**

## 📦 Celda 1 — Instalación de dependencias

In [ ]:
!pip install -q transformers==4.46.3 accelerate bitsandbytes sentencepiece rouge-score==0.1.2 bert-score==0.3.13

print('✅ Dependencias instaladas')

✅ Dependencias instaladas


## ⚙️ Celda 2 — Configuración

In [ ]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

# Carpeta en Drive donde estan el notebook y el dataset de entrada.
DRIVE_BASE           = '/content/drive/MyDrive/eval modelos base'
DRIVE_DATASET_PATH   = DRIVE_BASE + '/eval_dataset_150_cli_corrected.csv'
DRIVE_RESULTS_FOLDER = DRIVE_BASE + '/resultados'

MAX_NEW_TOKENS = 512

# Enfoques a ejecutar en secuencia.
# Para correr solo algunos, elimina o comenta las entradas que no necesites.
APPROACHES_TO_RUN = [
    {'name': 'zeroshot_directo',  'use_planning': False, 'apply_few_shot': False},
    {'name': 'fewshot_directo',   'use_planning': False, 'apply_few_shot': True},
    {'name': 'zeroshot_con_plan', 'use_planning': True,  'apply_few_shot': False},
    {'name': 'fewshot_con_plan',  'use_planning': True,  'apply_few_shot': True},
]

# Valores iniciales; el bucle principal los actualiza antes de cada enfoque.
USE_PLANNING = APPROACHES_TO_RUN[0]['use_planning']
APPLY_FEW_SHOT = APPROACHES_TO_RUN[0]['apply_few_shot']

# Modelos a evaluar (mismos paths que evaluate_generation.py).
MODELS = {
    'Llama-3.1-8B-Instruct': {
        'path': 'meta-llama/Llama-3.1-8B-Instruct',
        'params': '8B',
    },
    'Zephyr-7B': {
        'path': 'HuggingFaceH4/zephyr-7b-beta',
        'params': '7B',
    },
    'Qwen2.5-7B-Instruct': {
        'path': 'Qwen/Qwen2.5-7B-Instruct',
        'params': '7B',
    },
    'Gemma-2-9B-it': {
        'path': 'google/gemma-2-9b-it',
        'params': '9B',
    },
    'FLAN-T5-large': {
        'path': 'google/flan-t5-large',
        'params': '780M',
        'architecture': 'seq2seq',
    },
    'FLAN-T5-base': {
        'path': 'google/flan-t5-base',
        'params': '250M',
        'architecture': 'seq2seq',
    },
}

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME']  = '/root/.cache/huggingface'

print('Modelos a evaluar : ' + str(len(MODELS)))
for _k in MODELS:
    print('   - ' + _k)
print('Enfoques a ejecutar: ' + str(len(APPROACHES_TO_RUN)))
for _a in APPROACHES_TO_RUN:
    print('   - ' + _a['name'])
print('MAX_NEW_TOKENS    : ' + str(MAX_NEW_TOKENS))
print('Carpeta resultados: ' + DRIVE_RESULTS_FOLDER)



Modelos a evaluar : 6
   - Llama-3.1-8B-Instruct
   - Zephyr-7B
   - Qwen2.5-7B-Instruct
   - Gemma-2-9B-it
   - FLAN-T5-large
   - FLAN-T5-base
Modo              : con plan (2 pasos)
Few-shot          : activo
MAX_NEW_TOKENS    : 512
Carpeta resultados: /content/drive/MyDrive/eval modelos base/resultados


## 🔌 Celda 3 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

for path in [DRIVE_DATASET_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError('Archivo no encontrado: ' + path)
    print('OK ' + path)

os.makedirs(DRIVE_RESULTS_FOLDER, exist_ok=True)
print('OK Carpeta de resultados: ' + DRIVE_RESULTS_FOLDER)

Mounted at /content/drive
OK /content/drive/MyDrive/eval modelos base/eval_dataset_150_cli_corrected.csv
OK Carpeta de resultados: /content/drive/MyDrive/eval modelos base/resultados


## 🖥️ Celda 4 — Verificar GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('❌ CUDA no disponible. Activa GPU en Entorno de ejecucion.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print('✅ GPU: ' + gpu_name + '  |  VRAM: ' + '{:.1f}'.format(vram_gb) + ' GB')

if vram_gb < 14:
    print('⚠️  Menos de 14 GB. El modelo 4bit puede no caber.')

✅ GPU: NVIDIA L4  |  VRAM: 22.0 GB


## 📂 Celda 5 — Cargar dataset de evaluación

In [ ]:
import pandas as pd

# Dataset de evaluacion -> columnas: requirement, configuration.
raw = pd.read_csv(DRIVE_DATASET_PATH, encoding='utf-8')

for col in ['requirement', 'configuration']:
    assert col in raw.columns, 'Columna faltante: ' + col

df = pd.DataFrame({
    'requirement':  raw['requirement'].astype(str),
    'ground_truth': raw['configuration'].astype(str),
})
df['id'] = range(len(df))


def _count_config_lines(text):
    '''Cuenta lineas de configuracion reales del ground truth.
    Excluye prompts CLI y enable/configure terminal/end (igual criterio que normalize_config).
    '''
    n = 0
    for line in str(text).split(chr(10)):
        if '#' in line:
            line = line.split('#', 1)[-1]
        elif '>' in line:
            line = line.split('>', 1)[-1]
        line = ' '.join(line.split()).lower()
        if line and line not in ('configure terminal', 'end', 'enable', 'no_code'):
            n += 1
    return n


df['n_lines'] = df['ground_truth'].apply(_count_config_lines)

EVAL_SAMPLE_SIZE = None
EVAL_SAMPLE_SEED = 42

df['original_id'] = df['id']

if EVAL_SAMPLE_SIZE is not None and len(df) > EVAL_SAMPLE_SIZE:
    df = (
        df.sample(n=EVAL_SAMPLE_SIZE, random_state=EVAL_SAMPLE_SEED)
          .reset_index(drop=True)
    )
    df['id'] = range(len(df))

print('✅ Dataset cargado: ' + str(len(df)) + ' muestras')
print('   Columnas: ' + str(list(df.columns)))
print()
print('Distribucion n_lines:')
print(df['n_lines'].describe().to_string())
df.head(3)

✅ Dataset cargado: 150 muestras
   Columnas: ['requirement', 'ground_truth', 'id', 'n_lines', 'original_id']

Distribucion n_lines:
count    150.000000
mean       7.466667
std        7.214514
min        1.000000
25%        4.000000
50%        6.000000
75%        9.000000
max       54.000000


,requirement,ground_truth,id,n_lines,original_id
0,Configure R1 Ethernet0/1 to connect to R2 by a...,R1# configure terminal\nR1(config)# interface ...,0,3,0
1,Set up R2 Ethernet0/0 facing R1 with the 10.0....,R2# configure terminal\nR2(config)# interface ...,1,3,1
2,Configure both ends of the R1-R2 point-to-poin...,R1# configure terminal\nR1(config)# interface ...,2,6,2


## 📝 Celda 6 — Topología y prompts

In [ ]:
# -- Topologia de red (igual que topo_v1) ---------------------------------
NETWORK_CONTEXT = '''=== NETWORK TOPOLOGY ===

Interface table:
DEVICE  INTERFACE    IP ADDRESS    MASK             NEIGHBOR  NEIGHBOR IFACE
R1      Ethernet0/0  10.0.1.1      255.255.255.0    SW1       Fa0/24
R1      Ethernet0/1  10.0.12.1     255.255.255.0    R2        Ethernet0/0
R1      Ethernet0/2  10.0.14.1     255.255.255.0    R4        Ethernet0/0
R2      Ethernet0/0  10.0.12.2     255.255.255.0    R1        Ethernet0/1
R2      Ethernet0/1  10.0.23.1     255.255.255.0    R3        Ethernet0/0
R2      Ethernet0/2  10.0.24.1     255.255.255.0    R4        Ethernet0/1
R3      Ethernet0/0  10.0.23.2     255.255.255.0    R2        Ethernet0/1
R3      Ethernet0/1  10.0.34.1     255.255.255.0    R4        Ethernet0/2
R3      Ethernet0/2  10.0.2.1      255.255.255.0    SW2       Fa0/24
R4      Ethernet0/0  10.0.14.2     255.255.255.0    R1        Ethernet0/2
R4      Ethernet0/1  10.0.24.2     255.255.255.0    R2        Ethernet0/2
R4      Ethernet0/2  10.0.34.2     255.255.255.0    R3        Ethernet0/1

Hosts:
- h1: 10.0.1.10/24  gw 10.0.1.1   -> SW1 Fa0/1  (VLAN 10)
- h2: 10.0.2.10/24  gw 10.0.2.1   -> SW2 Fa0/1  (VLAN 30)

Switch ports:
- SW1 Fa0/1  : access VLAN 10, connected to h1
- SW1 Fa0/24 : trunk VLANs 10,20, connected to R1 Ethernet0/0
- SW2 Fa0/1  : access VLAN 30, connected to h2
- SW2 Fa0/24 : trunk VLANs 30,40, connected to R3 Ethernet0/2
'''

# -- Prompt de generacion: enfoque NoRAG (solo topologia) ------------------
GENERATION_PROMPT = '''You are a Cisco IOS XE network engineer. Generate the exact Cisco IOS XE configuration commands to fulfill the requirement, using the network topology.

RULES:
- Output ONLY configuration commands in Cisco IOS XE CLI format with mode prompts.
- Use the device name from the requirement as the CLI hostname (e.g., R1 -> R1(config)#, SW1 -> SW1(config)#).
- If no device name is specified, use Router(config)# or Switch(config)# as appropriate.
- Include correct sub-mode prompts: (config-if)#, (config-router)#, (config-line)#, (config-vlan)#, (config-router-af)#, (config-vrf)#, (config-ext-nacl)#, etc.
- No explanations, no markdown, no comments, no extra text.
- Use ONLY interfaces, IPs, VLANs and devices defined in the topology.

Network topology:
{network_context}

Requirement:
{requirement}

Configuration:'''

# -- Prompt de planificacion (flujo con plan) ------------------------------
PLANNING_PROMPT = '''You are a senior network engineer.

Identify the key points required to generate a correct Cisco IOS configuration.

Rules:

* Focus on key configuration points
* Output a concise bullet list using "- " at the start of each line
* Do NOT include Cisco IOS commands
* Include only points that are necessary and relevant to satisfy the requirement
* Cover critical dimensions when applicable: devices, interfaces, addressing, protocols, policy/ACL order, dependencies, and validation checks
* Do not invent values, devices, links, IPs, VLANs, or constraints outside the topology

Output format example:
- Key point one
- Key point two
- Key point three
- ...

Do not output explanations, comments, or markdown.

Topology:
{network_context}

Requirement:
{requirement}

Plan:
'''

# -- Prompt de generacion con plan: parte inicial = GENERATION_PROMPT ------
# (mismas reglas del flujo normal, sin ejemplos) + seccion de plan ---------
GENERATION_WITH_PLAN_PROMPT = '''You are a Cisco IOS XE network engineer. Generate the exact Cisco IOS XE configuration commands to fulfill the requirement, using the network topology.

RULES:
- Output ONLY configuration commands in Cisco IOS XE CLI format with mode prompts.
- Use the device name from the requirement as the CLI hostname (e.g., R1 -> R1(config)#, SW1 -> SW1(config)#).
- If no device name is specified, use Router(config)# or Switch(config)# as appropriate.
- Include correct sub-mode prompts: (config-if)#, (config-router)#, (config-line)#, (config-vlan)#, (config-router-af)#, (config-vrf)#, (config-ext-nacl)#, etc.
- No explanations, no markdown, no comments, no extra text.
- Use ONLY interfaces, IPs, VLANs and devices defined in the topology.

=== INPUT ===

Plan:
{plan}

Topology:
{network_context}

Requirement:
{requirement}

Configuration:
'''

# -- Prompt para modelos seq2seq / FLAN-T5 (de evaluate_generation.py) ------
GENERATION_PROMPT_SEQ2SEQ = '''Task: Generate Cisco IOS configuration commands.

Rules:
- Output only Cisco IOS commands.
- No explanations, notes, or markdown.
- Use only devices/interfaces/IPs/VLANs present in topology.
- If not possible with given topology, output exactly: NO_CODE.

Topology:
{network_context}

Requirement:
{requirement}

Answer:
'''


# -- Few-shot: 3 ejemplos en formato ground truth (con configure terminal/end)
FEW_SHOT_EXAMPLES = '''Requirement: Set the login banner on the router to display "Authorized access only".
Configuration:
Router# configure terminal
Router(config)# banner motd # Authorized access only #
Router(config)# end

Requirement: Configure the router to use the NTP server at 200.1.1.1 for time synchronization.
Configuration:
Router# configure terminal
Router(config)# ntp server 200.1.1.1
Router(config)# end

Requirement: Enable timestamped logging to the console on the switch.
Configuration:
Switch# configure terminal
Switch(config)# service timestamps log datetime
Switch(config)# logging console
Switch(config)# end'''

GENERATION_PROMPT_BASE = GENERATION_PROMPT
GENERATION_WITH_PLAN_PROMPT_BASE = GENERATION_WITH_PLAN_PROMPT
GENERATION_PROMPT_SEQ2SEQ_BASE = GENERATION_PROMPT_SEQ2SEQ


def configure_prompting(use_planning, apply_few_shot):
    global USE_PLANNING, APPLY_FEW_SHOT
    global GENERATION_PROMPT, GENERATION_WITH_PLAN_PROMPT, GENERATION_PROMPT_SEQ2SEQ

    USE_PLANNING = bool(use_planning)
    APPLY_FEW_SHOT = bool(apply_few_shot)

    GENERATION_PROMPT = GENERATION_PROMPT_BASE
    GENERATION_WITH_PLAN_PROMPT = GENERATION_WITH_PLAN_PROMPT_BASE
    GENERATION_PROMPT_SEQ2SEQ = GENERATION_PROMPT_SEQ2SEQ_BASE

    if APPLY_FEW_SHOT:
        _few_shot_block = '=== EXAMPLES ===' + chr(10) * 2 + FEW_SHOT_EXAMPLES.strip() + chr(10) * 2
        if USE_PLANNING:
            GENERATION_WITH_PLAN_PROMPT = GENERATION_WITH_PLAN_PROMPT.replace(
                '=== INPUT ===', _few_shot_block + '=== INPUT ===')
        else:
            GENERATION_PROMPT = GENERATION_PROMPT.replace(
                'Network topology:', _few_shot_block + 'Network topology:')
            GENERATION_PROMPT_SEQ2SEQ = GENERATION_PROMPT_SEQ2SEQ.replace(
                'Topology:', _few_shot_block + 'Topology:')

    print('Prompt configurado: ' + ('con plan' if USE_PLANNING else 'directo')
          + ' | ' + ('few-shot' if APPLY_FEW_SHOT else 'zero-shot'))


configure_prompting(USE_PLANNING, APPLY_FEW_SHOT)
print('OK Topologia y prompts definidos (causal + seq2seq + plan)')



Few-shot: ACTIVO (prompt con plan)
OK Topologia y prompts definidos (causal + seq2seq + plan)


## 🤖 Celda 7 — Funciones de carga de modelos (float16)

In [ ]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM,
)


def load_model(model_name, model_config):
    """Carga un modelo causal en float16 (mismo patron que evaluate_generation.py)."""
    try:
        if not torch.cuda.is_available():
            raise RuntimeError('CUDA no disponible. Activa GPU en Entorno de ejecucion.')

        device = 'cuda:0'
        print('\nCargando ' + model_name + ' en ' + torch.cuda.get_device_name(0) + '...')

        tokenizer = AutoTokenizer.from_pretrained(
            model_config['path'],
            trust_remote_code=model_config.get('trust_remote_code', False),
            token=HF_TOKEN,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_config['path'],
            torch_dtype=torch.float16,
            device_map=device,
            trust_remote_code=model_config.get('trust_remote_code', False),
            token=HF_TOKEN,
        )

        print('  VRAM usada: ' + '{:.2f}'.format(torch.cuda.memory_allocated(0) / 1024**3) + ' GB')
        print('  Modelo ' + model_name + ' cargado exitosamente')
        return tokenizer, model

    except Exception as e:
        print('Error cargando ' + model_name + ': ' + str(e))
        return None, None


def load_model_seq2seq(model_name, model_config):
    """Carga un modelo seq2seq / encoder-decoder en float16 (FLAN-T5)."""
    try:
        if not torch.cuda.is_available():
            raise RuntimeError('CUDA no disponible. Activa GPU en Entorno de ejecucion.')

        device = 'cuda:0'
        print('\nCargando ' + model_name + ' en ' + torch.cuda.get_device_name(0) + '...')

        tokenizer = AutoTokenizer.from_pretrained(
            model_config['path'],
            trust_remote_code=model_config.get('trust_remote_code', False),
            token=HF_TOKEN,
        )
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_config['path'],
            torch_dtype=torch.float16,
            device_map=device,
            trust_remote_code=model_config.get('trust_remote_code', False),
            token=HF_TOKEN,
        )

        print('  VRAM usada: ' + '{:.2f}'.format(torch.cuda.memory_allocated(0) / 1024**3) + ' GB')
        print('  Modelo ' + model_name + ' cargado exitosamente')
        return tokenizer, model

    except Exception as e:
        print('Error cargando ' + model_name + ': ' + str(e))
        return None, None


print('OK Funciones de carga definidas (causal + seq2seq, float16)')

OK Funciones de carga definidas (causal + seq2seq, float16)


## ⚙️ Celda 8 — Funciones de inferencia

In [ ]:
import re


def build_prompt(tokenizer, messages, plain_prompt):
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
    return plain_prompt


def generate_config(requirement, tokenizer, model, max_new_tokens=MAX_NEW_TOKENS):
    """Genera la configuracion (flujo causal, prompt IOS XE del notebook)."""
    try:
        plain = GENERATION_PROMPT.format(
            network_context=NETWORK_CONTEXT,
            requirement=requirement)
        messages = [{'role': 'user', 'content': plain}]
        prompt   = build_prompt(tokenizer, messages, plain)
        inputs   = tokenizer(
            prompt, return_tensors='pt', truncation=True, max_length=4096
        ).to('cuda:0')
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
                temperature=None, top_p=None, top_k=None)
        return tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    except Exception as e:
        print('  generate_config error: ' + str(e))
        return 'ERROR'


def generate_config_seq2seq(requirement, tokenizer, model, max_new_tokens=MAX_NEW_TOKENS):
    """Genera config para modelos seq2seq / FLAN-T5 (de evaluate_generation.py)."""
    try:
        prompt = GENERATION_PROMPT_SEQ2SEQ.format(
            requirement=requirement,
            network_context=NETWORK_CONTEXT,
        )
        inputs = tokenizer(
            prompt, return_tensors='pt', truncation=True, max_length=1024
        ).to('cuda:0')

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=min(max_new_tokens, 256),
                do_sample=False,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=4,
                length_penalty=0.8,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        generated = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

        stop_markers = [
            'Requirement:', 'Configuration:', 'Output:', 'Explanation:',
            'Note:', '===', 'Rules:', 'Task:', 'Topology:', 'Answer:',
        ]
        cut_positions = [generated.find(m) for m in stop_markers if generated.find(m) != -1]
        if cut_positions:
            generated = generated[: min(cut_positions)].strip()

        if re.search(r'\bNO_CODE\b', generated):
            has_ios_lines = bool(re.search(r'\b(config|interface|router|ip\s)\b', generated, re.IGNORECASE))
            if not has_ios_lines:
                return 'NO_CODE'
            generated = re.sub(r'\bOR\b\s*\bNO_CODE\b.*$', '', generated, flags=re.IGNORECASE | re.DOTALL).strip()

        if not generated:
            return 'NO_CODE'

        return generated

    except Exception as e:
        print('  generate_config_seq2seq error: ' + str(e))
        return 'ERROR'


def generate_with_plan(requirement, tokenizer, model):
    """Genera plan y luego la config final (2 inferencias). Soporta causal y seq2seq."""
    try:
        is_seq2seq = bool(getattr(getattr(model, 'config', None), 'is_encoder_decoder', False))

        # Paso 1: plan
        planning_prompt = PLANNING_PROMPT.format(
            requirement=requirement,
            network_context=NETWORK_CONTEXT,
        )

        if is_seq2seq:
            plan_inputs = tokenizer(
                planning_prompt, return_tensors='pt', truncation=True, max_length=1024
            ).to('cuda:0')
            with torch.no_grad():
                plan_outputs = model.generate(
                    **plan_inputs, max_new_tokens=128, do_sample=False,
                    num_beams=4, early_stopping=True, no_repeat_ngram_size=3,
                )
            plan = tokenizer.decode(plan_outputs[0], skip_special_tokens=True).strip()
        else:
            plan_messages = [{'role': 'user', 'content': planning_prompt}]
            plan_chat_prompt = tokenizer.apply_chat_template(
                plan_messages, tokenize=False, add_generation_prompt=True)
            plan_inputs = tokenizer(
                plan_chat_prompt, return_tensors='pt', truncation=True, max_length=2048
            ).to('cuda:0')
            with torch.no_grad():
                plan_outputs = model.generate(
                    **plan_inputs, max_new_tokens=128, do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    temperature=None, top_p=None, top_k=None,
                )
            plan = tokenizer.decode(
                plan_outputs[0][plan_inputs['input_ids'].shape[1]:],
                skip_special_tokens=True).strip()

        # Paso 2: config a partir del plan
        generation_prompt = GENERATION_WITH_PLAN_PROMPT.format(
            plan=plan,
            requirement=requirement,
            network_context=NETWORK_CONTEXT,
        )

        if is_seq2seq:
            cfg_inputs = tokenizer(
                generation_prompt, return_tensors='pt', truncation=True, max_length=1024
            ).to('cuda:0')
            with torch.no_grad():
                cfg_outputs = model.generate(
                    **cfg_inputs, max_new_tokens=256, do_sample=False,
                    num_beams=4, early_stopping=True,
                    no_repeat_ngram_size=4, length_penalty=0.8,
                )
            generated = tokenizer.decode(cfg_outputs[0], skip_special_tokens=True).strip()
        else:
            cfg_messages = [{'role': 'user', 'content': generation_prompt}]
            cfg_chat_prompt = tokenizer.apply_chat_template(
                cfg_messages, tokenize=False, add_generation_prompt=True)
            cfg_inputs = tokenizer(
                cfg_chat_prompt, return_tensors='pt', truncation=True, max_length=2048
            ).to('cuda:0')
            with torch.no_grad():
                cfg_outputs = model.generate(
                    **cfg_inputs, max_new_tokens=256, do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    temperature=None, top_p=None, top_k=None,
                )
            generated = tokenizer.decode(
                cfg_outputs[0][cfg_inputs['input_ids'].shape[1]:],
                skip_special_tokens=True).strip()

        # Limpieza: cortar en markers de prompt
        stop_markers = [
            'Plan:', 'Topology:', 'Requirement:', 'Configuration:',
            'Output:', 'Explanation:', 'Note:', '===',
        ]
        cut_positions = [generated.find(m) for m in stop_markers if generated.find(m) != -1]
        if cut_positions:
            generated = generated[:min(cut_positions)].strip()

        if re.search(r'\bNO_CODE\b', generated, re.IGNORECASE):
            has_ios = bool(re.search(
                r'\b(config|interface|router|ip\s|switchport|access-list|route-map|vlan|spanning-tree|line\s+vty|hostname|enable|copy\s+running-config|end)\b',
                generated, re.IGNORECASE))
            if not has_ios:
                generate_with_plan.last_plan = plan
                return 'NO_CODE'
            generated = re.sub(r'\bOR\b\s*\bNO_CODE\b.*$', '', generated, flags=re.IGNORECASE | re.DOTALL).strip()

        if not generated:
            generate_with_plan.last_plan = plan
            return 'NO_CODE'

        generate_with_plan.last_plan = plan
        return generated

    except Exception as e:
        print('  generate_with_plan error: ' + str(e))
        generate_with_plan.last_plan = ''
        return 'NO_CODE'


print('OK Funciones de inferencia definidas (causal + seq2seq + plan)')

OK Funciones de inferencia definidas (causal + seq2seq + plan)


## 📊 Celda 9 — Funciones de métricas (ROUGE normalizado + BERTScore)

In [ ]:
import numpy as np


def normalize_config(text):
    '''Elimina prompts CLI, configure terminal y end para comparacion justa.
    El prompt v3 genera configure terminal/end; se eliminan aqui para que
    el ROUGE sea comparable con el ground truth de topo_v1 que no los tiene.
    '''
    lines = []
    for line in text.split(chr(10)):
        if '#' in line:
            line = line.split('#', 1)[-1]
        elif '>' in line:
            line = line.split('>', 1)[-1]
        line = ' '.join(line.split()).lower()
        if line and line not in ('configure terminal', 'end', 'enable', 'no_code'):
            lines.append(line)
    return lines


def compute_rouge(predictions, references):
    from rouge_score import rouge_scorer as rs
    scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        if not pred or not ref or pred == 'ERROR':
            continue
        pred_norm = ' '.join(normalize_config(pred))
        ref_norm  = ' '.join(normalize_config(ref))
        if not pred_norm or not ref_norm:
            continue
        s = scorer.score(ref_norm, pred_norm)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rl.append(s['rougeL'].fmeasure)

    def safe(lst):    return round(float(np.mean(lst)), 4) if lst else 0.0
    def safestd(lst): return round(float(np.std(lst)),  4) if lst else 0.0

    return {
        'rouge1': safe(r1), 'rouge1_std': safestd(r1),
        'rouge2': safe(r2), 'rouge2_std': safestd(r2),
        'rougeL': safe(rl), 'rougeL_std': safestd(rl),
    }


def compute_bertscore(predictions, references):
    from bert_score import score as bscore
    valid = [(p, r) for p, r in zip(predictions, references)
             if p and r and p != 'ERROR']
    if not valid:
        return {'bertscore_p': 0.0, 'bertscore_r': 0.0,
                'bertscore_f1': 0.0, 'bertscore_f1_std': 0.0}
    preds, refs = zip(*valid)
    for model_name, num_layers in [('microsoft/codebert-base', 12), ('roberta-large', None)]:
        try:
            kwargs = {'lang': 'en', 'model_type': model_name,
                      'verbose': False, 'batch_size': 16}
            if num_layers:
                kwargs['num_layers'] = num_layers
            P, R, F1 = bscore(list(preds), list(refs), **kwargs)
            print('    BERTScore model: ' + model_name)
            return {
                'bertscore_p':      round(float(P.mean()),  4),
                'bertscore_r':      round(float(R.mean()),  4),
                'bertscore_f1':     round(float(F1.mean()), 4),
                'bertscore_f1_std': round(float(F1.std()),  4),
            }
        except Exception as e:
            print('    ⚠️ ' + model_name + ': ' + str(e))
    return {'bertscore_p': 0.0, 'bertscore_r': 0.0,
            'bertscore_f1': 0.0, 'bertscore_f1_std': 0.0}


print('✅ Metricas definidas (ROUGE normalizado + BERTScore)')

✅ Metricas definidas (ROUGE normalizado + BERTScore)


## 🚀 Celda 10 — Evaluación de los 6 modelos (bucle único)

In [ ]:
import time
import gc
import json
from datetime import datetime


def evaluate_model(model_name, model_config, df_eval):
    """Carga, evalua y libera un modelo. Devuelve dict de resultados o None."""
    print('\n' + '=' * 80)
    print('EVALUANDO: ' + model_name + ' (' + model_config['params'] + ')')
    print('=' * 80)

    architecture = model_config.get('architecture', 'causal')
    if architecture == 'seq2seq':
        tokenizer, model = load_model_seq2seq(model_name, model_config)
        generation_fn = generate_config_seq2seq
    else:
        tokenizer, model = load_model(model_name, model_config)
        generation_fn = generate_config

    if tokenizer is None or model is None:
        print('Saltando ' + model_name + ' - no se pudo cargar.')
        return None

    try:
        predictions = []
        plans       = []
        latencies   = []
        n = len(df_eval)

        for i, (_, row) in enumerate(df_eval.iterrows()):
            if i % 20 == 0:
                vram = torch.cuda.memory_allocated(0) / 1024**3
                print('  [' + str(i + 1).rjust(3) + '/' + str(n) + ']  VRAM: ' + '{:.2f}'.format(vram) + ' GB')

            t0 = time.time()
            if USE_PLANNING:
                pred = generate_with_plan(row['requirement'], tokenizer, model)
                plans.append(getattr(generate_with_plan, 'last_plan', ''))
            else:
                pred = generation_fn(row['requirement'], tokenizer, model)
                plans.append('')
            latencies.append(time.time() - t0)
            predictions.append(pred)

        references   = df_eval['ground_truth'].tolist()
        error_count  = predictions.count('ERROR')
        nocode_count = sum(1 for p in predictions if isinstance(p, str) and p.strip().upper() == 'NO_CODE')

        print('\n  Calculando ROUGE (normalizado)...')
        rouge_metrics = compute_rouge(predictions, references)
        print('  Calculando BERTScore...')
        bert_metrics = compute_bertscore(predictions, references)

        # Desglose por complejidad (n_lines del ground truth).
        complexity_results = {}
        for label, mask in [
            ('corta  (<=3)', df_eval['n_lines'].astype(int) <= 3),
            ('media  (4-7)', (df_eval['n_lines'].astype(int) >= 4) & (df_eval['n_lines'].astype(int) <= 7)),
            ('larga  (>7) ', df_eval['n_lines'].astype(int) > 7),
        ]:
            idx     = [df_eval.index.get_loc(j) for j in df_eval.index[mask]]
            preds_s = [predictions[k] for k in idx]
            refs_s  = [references[k]  for k in idx]
            r = compute_rouge(preds_s, refs_s)
            complexity_results[label] = {**r, 'n': len(idx)}

        total_time = sum(latencies)
        avg_time   = float(np.mean(latencies))

        results = {
            'model_name':          model_name,
            'params':              model_config['params'],
            'architecture':        architecture,
            'use_planning':        USE_PLANNING,
            'apply_few_shot':      APPLY_FEW_SHOT,
            'samples_evaluated':   n,
            'error_count':         int(error_count),
            'nocode_count':        int(nocode_count),
            'total_time_s':        round(total_time, 2),
            'avg_time_per_sample': round(avg_time, 4),
            **rouge_metrics,
            **bert_metrics,
            'by_complexity':       complexity_results,
            'plans':               plans,
            'predictions':         predictions,
        }

        print('\n  Resultados ' + model_name + ':')
        print('    ROUGE-1:        ' + '{:.4f}'.format(rouge_metrics['rouge1']) + '  (std ' + '{:.4f}'.format(rouge_metrics['rouge1_std']) + ')')
        print('    ROUGE-2:        ' + '{:.4f}'.format(rouge_metrics['rouge2']) + '  (std ' + '{:.4f}'.format(rouge_metrics['rouge2_std']) + ')')
        print('    ROUGE-L:        ' + '{:.4f}'.format(rouge_metrics['rougeL']) + '  (std ' + '{:.4f}'.format(rouge_metrics['rougeL_std']) + ')')
        print('    BERTScore-F1:   ' + '{:.4f}'.format(bert_metrics['bertscore_f1']) + '  (std ' + '{:.4f}'.format(bert_metrics['bertscore_f1_std']) + ')')
        print('    Tiempo/muestra: ' + '{:.3f}'.format(avg_time) + 's')
        print('    Errores:        ' + str(error_count) + '/' + str(n) + '  NO_CODE: ' + str(nocode_count))

        return results

    finally:
        del model
        del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            print('  VRAM post-modelo: allocada ' + '{:.2f}'.format(torch.cuda.memory_allocated(0) / 1024**3)
                  + ' GB | reservada ' + '{:.2f}'.format(torch.cuda.memory_reserved(0) / 1024**3) + ' GB')


references = df['ground_truth'].tolist()
all_run_outputs = []

print('=' * 80)
print('EVALUACION DE GENERACION DE CONFIGURACIONES CISCO')
print('Inicio: ' + datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Dataset: ' + str(len(df)) + ' muestras  |  Modelos: ' + str(len(MODELS)))
print('Enfoques: ' + str(len(APPROACHES_TO_RUN)) + '  |  float16')
print('=' * 80)

for approach in APPROACHES_TO_RUN:
    configure_prompting(approach['use_planning'], approach['apply_few_shot'])

    print('\n' + '#' * 80)
    print('ENFOQUE: ' + approach['name'])
    print('Modo: ' + ('con plan' if USE_PLANNING else 'directo')
          + '  |  ' + ('few-shot' if APPLY_FEW_SHOT else 'zero-shot')
          + '  |  float16')
    print('#' * 80)

    run_results = []
    for model_name, model_config in MODELS.items():
        result = evaluate_model(model_name, model_config, df)
        if result:
            run_results.append(result)

    timestamp       = datetime.now().strftime('%Y%m%d_%H%M%S')
    planning_suffix = '_con_plan' if USE_PLANNING else '_sin_plan'
    fewshot_suffix  = '_fewshot' if APPLY_FEW_SHOT else '_zeroshot'
    results_file    = 'generation_results' + planning_suffix + fewshot_suffix + '_' + timestamp + '.json'
    output_path     = os.path.join(DRIVE_RESULTS_FOLDER, results_file)

    output = {
        'timestamp':        timestamp,
        'approach_name':    approach['name'],
        'dataset':          DRIVE_DATASET_PATH,
        'total_samples':    len(df),
        'use_planning':     USE_PLANNING,
        'apply_few_shot':   APPLY_FEW_SHOT,
        'dtype':            'float16',
        'models_evaluated': list(MODELS.keys()),
        'references':       references,
        'results':          run_results,
        'output_path':      output_path,
    }

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    all_run_outputs.append(output)

    print('\n' + '=' * 80)
    print('RESULTADOS GUARDADOS EN: ' + output_path)
    print('=' * 80)

    print('\n' + ('COMPARACION FINAL - ' + approach['name']).center(80))
    print('{:<26}{:<10}{:<10}{:<10}{:<12}{}'.format('Modelo', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERT-F1', 'T/muestra'))
    print('-' * 80)
    for r in sorted(run_results, key=lambda x: x['bertscore_f1'], reverse=True):
        print('{:<26}{:<10.4f}{:<10.4f}{:<10.4f}{:<12.4f}{:.3f}s'.format(
            r['model_name'], r['rouge1'], r['rouge2'], r['rougeL'], r['bertscore_f1'], r['avg_time_per_sample']))

# Mantiene compatibilidad con la celda de inspeccion: apunta al ultimo enfoque ejecutado.
all_results = all_run_outputs[-1]['results'] if all_run_outputs else []

print('\n' + '=' * 80)
print('EJECUCIONES COMPLETADAS: ' + str(len(all_run_outputs)))
for output in all_run_outputs:
    print(' - ' + output['approach_name'] + ': ' + output['output_path'])
print('=' * 80)



EVALUACION DE GENERACION DE CONFIGURACIONES CISCO
Modo: con plan  |  float16
Inicio: 2026-07-05 17:20:53
Dataset: 150 muestras  |  Modelos: 6

EVALUANDO: Llama-3.1-8B-Instruct (8B)

Cargando Llama-3.1-8B-Instruct en NVIDIA L4...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

  VRAM usada: 0.55 GB
  Modelo FLAN-T5-base cargado exitosamente
  [  1/150]  VRAM: 0.55 GB
  [ 21/150]  VRAM: 0.55 GB
  [ 41/150]  VRAM: 0.55 GB
  [ 61/150]  VRAM: 0.55 GB
  [ 81/150]  VRAM: 0.55 GB
  [101/150]  VRAM: 0.55 GB
  [121/150]  VRAM: 0.55 GB
  [141/150]  VRAM: 0.55 GB

  Calculando ROUGE (normalizado)...
  Calculando BERTScore...
    BERTScore model: microsoft/codebert-base

  Resultados FLAN-T5-base:
    ROUGE-1:        0.2109  (std 0.1773)
    ROUGE-2:        0.0866  (std 0.0941)
    ROUGE-L:        0.1822  (std 0.1479)
    BERTScore-F1:   0.8554  (std 0.0282)
    Tiempo/muestra: 5.135s
    Errores:        0/150  NO_CODE: 0
  VRAM post-modelo: allocada 0.01 GB | reservada 0.02 GB

RESULTADOS GUARDADOS EN: /content/drive/MyDrive/eval modelos base/resultados/generation_results_con_plan_fewshot_20260705_211218.json

                               COMPARACION FINAL                                
Modelo                    ROUGE-1   ROUGE-2   ROUGE-L   BERT-F1     T/muestra
--

## 🔍 Celda 11 — Inspección de predicciones

In [ ]:
INSPECT_MODEL = all_results[0]['model_name'] if all_results else None
IDX = 0

res = next((r for r in all_results if r['model_name'] == INSPECT_MODEL), None)
if res is None:
    print('No hay resultados para inspeccionar.')
else:
    row = df.iloc[IDX]
    print('MODELO        : ' + res['model_name'])
    print('ID            : ' + str(row['id']))
    print('n_lines       : ' + str(row['n_lines']))
    print(chr(10) + '📌 REQUIREMENT:')
    print(row['requirement'])
    if res['use_planning'] and res['plans'][IDX]:
        print(chr(10) + '🧭 PLAN:')
        print(res['plans'][IDX])
    print(chr(10) + '🤖 PREDICCION:')
    print(res['predictions'][IDX])
    print(chr(10) + '✅ GROUND TRUTH:')
    print(references[IDX])
    r1_val = compute_rouge([res['predictions'][IDX]], [references[IDX]])['rouge1']
    print(chr(10) + 'ROUGE-1 (normalizado): ' + '{:.4f}'.format(r1_val))

MODELO        : Llama-3.1-8B-Instruct
ID            : 0
n_lines       : 3

📌 REQUIREMENT:
Configure R1 Ethernet0/1 to connect to R2 by assigning it the 10.0.12.1/24 address and bringing the interface up.

🧭 PLAN:
- Define the interface to be configured as Ethernet0/1 on R1.
- Assign the IP address 10.0.12.1 to the interface.
- Assign the subnet mask 255.255.255.0 to the interface.
- Enable the interface.
- Verify the interface is up and has the correct IP address.
- Ensure the neighbor R2 is reachable via the interface.
- Verify the neighbor R2's interface Ethernet0/0 is up and has the correct IP address 10.0.12.2.
- Ensure the routing protocol is configured to advertise the network 10.0.12.0

🤖 PREDICCION:
R1# configure terminal
R1(config)# interface Ethernet0/1
R1(config-if)# ip address 10.0.12.1 255.255.255.0
R1(config-if)# no shutdown
R1(config-if)# exit
R1(config)# ip route 10.0.12.0 255.255.255.0 10.0.12.2
R1(config)# end
R1# show ip interface Ethernet0/1
R1# show ip route 10.0.1

## ⏹️ Celda 12 — Desconectar la sesión (al final de todo)

In [ ]:
# Libera la GPU y desconecta el entorno de ejecucion de Colab.
# Ejecutar SOLO cuando ya se guardaron los resultados en Drive.
from google.colab import runtime
runtime.unassign()